In [2]:
"""
TIER 1 ONLY: entity family tagging.

What this does, in plain terms:
  Groups entities that are LOOSELY similar into a shared "family_id".
  This does NOT change your entity names, and does NOT merge any nodes.
  It just adds a new column you can filter/search on.

  Example: "soy protein isolate", "soybean protein isolate", "soy flour"
  might all get family_id = 3. You can now pull all three with one
  filter, but they remain three separate entities everywhere else.

Why the threshold is loose (0.40 distance, looser than other scripts):
  Getting this wrong in the "too many things grouped together" direction
  costs you almost nothing, you just see an extra irrelevant row when
  filtering. Getting it wrong in the "missed a real connection" direction
  costs you real coverage. So this script is deliberately generous.

This is the ONLY tier that changes nothing about your actual data,
it is purely additive. Safe to run and inspect without any downstream
consequences.

Requires OPENAI_API_KEY (embeddings only, no LLM call in this script).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "expanded_triples.xlsx"  # swap to full corpus file when ready
SOURCE_COL = "expanded_source"
TARGET_COL = "expanded_target" 

EMBED_MODEL = "text-embedding-3-small"
DISTANCE_THRESHOLD = 0.40   # loose on purpose, see docstring

OUTPUT_XLSX = "tier1_entity_family_tags.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# LOAD ENTITIES
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)
entities = sorted(pd.concat([ 
    df[SOURCE_COL].astype(str).str.strip(),
    df[TARGET_COL].astype(str).str.strip(),
]).unique())
print(f"Unique entities: {len(entities)}")

# ---------------------------------------------------------------
# EMBED
# ---------------------------------------------------------------
def embed_batch(texts, batch_size=200):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
        time.sleep(0.2)
    return np.array(vectors)


embeddings = embed_batch(entities)
print(f"Embedded {embeddings.shape[0]} entities")

# ---------------------------------------------------------------
# CLUSTER INTO FAMILIES (loose threshold)
# ---------------------------------------------------------------
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=DISTANCE_THRESHOLD,
    metric="cosine",
    linkage="average",
)
family_ids = clustering.fit_predict(embeddings)

family_df = pd.DataFrame({"entity": entities, "family_id": family_ids})
family_df = family_df.sort_values(["family_id", "entity"]).reset_index(drop=True)

family_sizes = family_df["family_id"].value_counts()
print(f"\nFormed {family_df['family_id'].nunique()} families")
print(f"  Singletons (no family found): {(family_sizes == 1).sum()}")
print(f"  Families with 2+ members: {(family_sizes >= 2).sum()}")
print(f"  Largest family size: {family_sizes.max()}")

print("\nExample families (first 5 multi-member ones):")
shown = 0
for fid, group in family_df.groupby("family_id"):
    if len(group) >= 2:
        print(f"  family_id {fid}: {group['entity'].tolist()}")
        shown += 1
    if shown >= 5:
        break

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
family_df.to_excel(OUTPUT_XLSX, index=False)
print(f"\nSaved to {OUTPUT_XLSX}")
print("Nothing in your main triples file has been changed. This is an")
print("additive lookup table: entity -> family_id, for filtering only.")

Unique entities: 3100
Embedded 3100 entities

Formed 798 families
  Singletons (no family found): 293
  Families with 2+ members: 505
  Largest family size: 136

Example families (first 5 multi-member ones):
  family_id 0: ['acid soluble protein concentrate', 'acid-soluble pinto bean protein concentrate', 'acid-soluble pinto bean protein isolate fraction', 'alkali solution with isoelectric precipitation peanut protein concentrate', 'precipitated and acid-soluble pinto bean protein concentrates', 'precipitated pinto bean protein concentrate']
  family_id 1: ['bound water content', 'free water content', 'overall water-holding capacity', 'restricted water content', 'water absorption', 'water absorption capacity', 'water absorption index', 'water activity', 'water binding', 'water content', 'water holding capacity', 'water holding capacity of the insoluble fraction', 'water hydration capacity', 'water imbibition capacity', 'water interaction capacity', 'water retention', 'water retention c